# 문제 1-1 : 기본 Chain 만들기 — AI 요리사

사용자가 재료를 입력하면 그 재료로 만들 수 있는 요리와 간단한 레시피를 추천한다

## 1. 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정 (.env 는 프로젝트 루트에 있음)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 전체는 출력하지 않고 설정 여부만 확인
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

## 2. LLM 설정 (ChatOpenAI)

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq"(무료) 또는 "openai" — 이 값만 바꾸면 아래 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# Groq는 OpenAI 호환 API라서 ChatOpenAI에 base_url만 바꿔서 사용
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}


def get_llm(temperature: float = 0.7, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자의 ChatOpenAI 인스턴스를 생성합니다."""
    config = LLM_CONFIGS[provider or PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {(provider or PROVIDER).upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, max_retries=3)


llm = get_llm()
print(f"{PROVIDER} / {llm.model_name}")

## 3. PromptTemplate 작성
`{ingredients}` 자리에 사용자가 입력한 재료가 들어간다.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """당신은 집에 있는 재료로 쉽게 만들 수 있는 요리를 추천해 주는 친절한 AI 요리사입니다.
아래 재료로 만들 수 있는 요리 1가지를 추천하고, 간단한 레시피를 알려주세요.

재료: {ingredients}

다음 형식을 지켜서 한국어로 답변하세요.
{ingredients}(으)로 만들 수 있는 요리를 추천드립니다!
추천 요리: (요리 이름)
재료: (필요한 재료 목록, 입력에 없는 기본 양념은 '추가 재료'로 구분)
조리법:
1. ...
2. ...
3. ...
"""

prompt = PromptTemplate.from_template(template)

print(prompt.input_variables)  # ['ingredients']
print(prompt.format(ingredients="계란, 밥, 김치"))

## 4. LCEL 체인 연결: `prompt | llm | StrOutputParser()`

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
print(type(chain))  # RunnableSequence

## 5. 실행 — 예시 입력 `"계란, 밥, 김치"`

In [ ]:
result = chain.invoke({"ingredients": "계란, 밥, 김치"})

print(type(result))  # StrOutputParser 덕분에 AIMessage 가 아니라 str
print(result)

## 6. 다른 재료로 테스트 — `"토마토, 양파, 치즈"` (stream 출력)

In [ ]:
for token in chain.stream({"ingredients": "토마토, 양파, 치즈"}):
    print(token, end="", flush=True)

## 7. (선택) 직접 재료 입력해 보기

In [ ]:
ingredients = input("가지고 있는 재료를 콤마(,)로 구분해 입력하세요: ").strip()

if ingredients:
    print(chain.invoke({"ingredients": ingredients}))
else:
    print("재료가 입력되지 않았습니다.")